# Observability with hooks

_Add logging, write a custom hook, and send traces to Langfuse._

⏱️ **~20 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom.


## Setup

In [ ]:
# Install JAZ from PyPI
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ routes to providers through LiteLLM and reads each provider's standard
# API-key environment variable. Set at least one. In Colab you can also use the
# secrets manager (key icon in the left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz
from jaz.llm import LiteLLM

# Pick the model all agents use. The model id is a LiteLLM route, 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))


## Logging

Hooks activate as context managers and propagate to nested invokes. Start with a
logger to see what the agent is doing.

In [ ]:
from jaz import invoke
from jaz.hooks import ReturnType, PrintLogger

with PrintLogger():
    result = invoke(ReturnType(int), task="Calculate 2 + 2")
result


## A custom hook

A hook observes **events** and returns **effects**. Here we nudge the agent to be
concise after a few iterations.

In [ ]:
from jaz.hooks import Hook
from jaz.hooks.events import LLMQueryEnter
from jaz.hooks.effects import Effect, AddMessages

class ConciseHook(Hook):
    """Nudge the agent to wrap up after a few iterations."""

    def on_llm_query_enter(self, event: LLMQueryEnter) -> list[Effect]:
        if event.iteration > 3:
            return [AddMessages([
                {"role": "user", "content": "Be concise — you're low on iterations."}
            ])]
        return []

with ConciseHook():
    result = invoke(ReturnType(int), task="Solve this step by step: 12 * 11")
result


## Distributed tracing (optional)

Install the tracing extra (`pip install 'jaz-lang[tracing]'`) and set your Langfuse
keys to send traces to Langfuse Cloud.

```python
from jaz import invoke
from jaz.hooks import LangfuseTracing, ReturnType

with LangfuseTracing():  # needs LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY
    result = invoke(ReturnType(int), task="Calculate 2 + 2")
```


That's the tour! Browse the [reference docs](https://jaz-lang.ai/docs/reference/hooks)
for every event and effect.